# Week 8: Networks II: Transit

The other networked structures we deal with often in urban data science are transit maps. Like roads, each transit line can be represented by a `LineString`, and each stop is a point along it. When transit stops have multiple lines running through it, it can become a network. 

Transit data is often stored for most cities in a format called General Transit Feed Specfication (GTFS). This format encodes the stops, lines, route schedules, and agency information in one place. 

For today, I have pre-downloaded the GTFS for Boston, on Canvas under Data as `MBTA_GTFS.zip`. Save this into the same folder as this notebook.

## 1 Getting to GTFS static data

Often times we want the transit schedules and stop locations as they are printed on the map in each city. This is called **static** data. It doesn't change day to day unless the agency produces an update. 

GTFS feeds for most cities are provided at https://mobilitydatabase.org/ or https://www.transit.land/feeds or often at the agency's website itself (for example this one for Boston: https://www.mbta.com/developers/gtfs). 

They are saved as .zip files containing the following:
* agency.txt — required, transit agencies with service represented in this dataset
* stops.txt — required, stops where vehicles pick up or drop off riders. Also defines stations and station entrances
* routes.txt — required, transit routes. A route is a group of trips that are displayed to riders as a single service
* trips.txt — required, trips for each route. A trip is a sequence of two or more stops that occur during a specific time period
* stop_times.txt — required, times that a vehicle arrives at and departs from stops for each trip
* calendar.txt — conditionally required, service dates specified using a weekly schedule with start and end dates
* calendar_dates.txt — conditionally required, exceptions for the services defined in the calendar.txt
* feed_info.txt — optional, dataset metadata, including publisher, version, and expiration information

Here's an [example](https://gtfs.org/getting-started/example-feed/) of what the inside of the .txt files look like. 

## 2 Reading in GTFS

There are two ways I recommend reading in the GTFS zip files:
1. Unzip the folder and read in each .txt file to pandas
2. Use a prebuilt package called `gtfs_kit`

Let's walk through the benefits and cons of each.

### 2.1 Reading in .txt files


In [11]:
import pandas as pd
import geopandas as gpd

In [36]:
# There are a few main files we might care about
# Each stop as a row
stops = pd.read_csv('MBTA_GTFS/stops.txt')
stops.head()

,stop_id,stop_code,stop_name,stop_desc,platform_code,platform_name,stop_lat,stop_lon,zone_id,stop_address,stop_url,level_id,location_type,parent_station,wheelchair_boarding,municipality,on_street,at_street,vehicle_type
0,1,1.0,Washington St opp Ruggles St,NaN,NaN,NaN,42.330957,-71.082754,ExpressBus-Downtown,NaN,https://www.mbta.com/stops/1,NaN,0,NaN,1,Boston,Washington Street,Ruggles Street,3.0
1,10,10.0,Theo Glynn Way @ Newmarket Sq,NaN,NaN,NaN,42.330555,-71.068787,LocalBus,NaN,https://www.mbta.com/stops/10,NaN,0,NaN,1,Boston,Theodore Glynn Way,Newmarket Square,3.0
2,10000,10000.0,Tremont St opp Temple Pl,NaN,NaN,NaN,42.355692,-71.062911,LocalBus,NaN,https://www.mbta.com/stops/10000,NaN,0,NaN,1,Boston,Tremont Street,Temple Place,3.0
3,10003,10003.0,Albany St opp Randall St,NaN,NaN,NaN,42.331591,-71.076237,LocalBus,NaN,https://www.mbta.com/stops/10003,NaN,0,NaN,1,Boston,Albany Street,Randall Street,3.0
4,10005,10005.0,Albany St opp E Concord St,NaN,NaN,NaN,42.335017,-71.071280,LocalBus,NaN,https://www.mbta.com/stops/10005,NaN,0,NaN,1,Boston,Albany Street,NaN,3.0


In [16]:
# Each named route in a transit system
routes = pd.read_csv('MBTA_GTFS/routes.txt')
routes.head()

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color,route_sort_order,route_fare_class,line_id,listed_route,network_id
0,Red,1,NaN,Red Line,Rapid Transit,1,https://www.mbta.com/schedules/Red,DA291C,FFFFFF,10010,Rapid Transit,line-Red,NaN,rapid_transit
1,Mattapan,1,NaN,Mattapan Line,Rapid Transit,0,https://www.mbta.com/schedules/Mattapan,DA291C,FFFFFF,10011,Rapid Transit,line-Mattapan,NaN,m_rapid_transit
2,Orange,1,NaN,Orange Line,Rapid Transit,1,https://www.mbta.com/schedules/Orange,ED8B00,FFFFFF,10020,Rapid Transit,line-Orange,NaN,rapid_transit
3,Green-B,1,B,Green Line B,Rapid Transit,0,https://www.mbta.com/schedules/Green-B,00843D,FFFFFF,10032,Rapid Transit,line-Green,NaN,rapid_transit
4,Green-C,1,C,Green Line C,Rapid Transit,0,https://www.mbta.com/schedules/Green-C,00843D,FFFFFF,10033,Rapid Transit,line-Green,NaN,rapid_transit


In [17]:
# The geometry of each route
shapes = pd.read_csv('MBTA_GTFS/shapes.txt')
shapes.head()

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,010140,42.373040,-71.117686,10001,NaN
1,010140,42.373096,-71.117908,10002,NaN
2,010140,42.373269,-71.118463,10003,NaN
3,010140,42.373353,-71.118582,10004,NaN
4,010140,42.373237,-71.118760,10005,NaN


In [15]:
# Each trip of a route
trips = pd.read_csv('MBTA_GTFS/trips.txt')
trips.head()

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,trip_route_type,route_pattern_id,bikes_allowed
0,1,BUS12026-hbc16fr1-Weekday-02,73542513,Harvard,NaN,0,C01-7,010154,1,NaN,1-_-0,1
1,1,BUS12026-hbc16fr1-Weekday-02,73542518,Harvard,NaN,0,C01-8,010154,1,NaN,1-_-0,1
2,1,BUS12026-hbc16fr1-Weekday-02,73542519,Harvard,NaN,0,C01-9,010154,1,NaN,1-_-0,1
3,1,BUS12026-hbc16fr1-Weekday-02,73542520,Harvard,NaN,0,C01-7,010154,1,NaN,1-_-0,1
4,1,BUS12026-hbc16fr1-Weekday-02,73542522,Harvard,NaN,0,C01-2,010154,1,NaN,1-_-0,1


In [35]:
# For each trip, the information about when the transit vehicle comes and goes to a stop
stop_times = pd.read_csv('MBTA_GTFS/stop_times.txt')
stop_times.head()

/var/folders/p3/nl4g9f296vj7y7kcrk8f76380000gn/T/ipykernel_58008/916012036.py:2: DtypeWarning: Columns (0: trip_id, 1: stop_id, 2: stop_headsign) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv('MBTA_GTFS/stop_times.txt')


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,timepoint,checkpoint_id,continuous_pickup,continuous_drop_off
0,72758838,06:00:00,06:00:00,70036,1,NaN,0,1,0,ogmnl,NaN,NaN
1,72758838,06:01:00,06:01:00,70034,10,NaN,0,0,0,mlmnl,NaN,NaN
2,72758838,06:05:00,06:05:00,70032,20,NaN,0,0,0,welln,NaN,NaN
3,72758838,06:07:00,06:07:00,70278,30,NaN,0,0,0,astao,NaN,NaN
4,72758838,06:09:00,06:09:00,70030,40,NaN,0,0,0,sull,NaN,NaN


So to recap, here are some important connector columns between the files: 
* `route_id`: connects `routes` and `trips`
* `trip_id`: connects `trips` and `stop_times`
* `shape_id`: connects `shapes` and `trips`
* `stop_id`: connects `stops` and `stop_times`

### 2.2 Installing gtfs_kit

`gtfs_kit` is a repbuilt package that has some functionality built in to it

In [3]:
!pip install gtfs_kit

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached rtree-1.4.1-py3-none-macosx_10_9_x86_64.whl.metadata (2.1 kB)
  Created wheel for json2html: filename=json2html-1.3.0-py3-none-any.whl size=7665 sha256=16b946f73e27c5a35f9a1005b03f9364881e13dbd3f51c3b12de46a8389d5e3b
  Stored in directory: /Users/madilore/Library/Caches/pip/wheels/b5/f4/39/ec8f0634aa27729654e0913b9c84d0ab7569bbcff7df0457e1
Successfully built json2html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gtfs_kit]


In [4]:
import gtfs_kit as gkt

In [6]:
# Now we read in directly from the zip file and all of the files are taken in
feed = gkt.read_feed("MBTA_GTFS.zip", dist_units="ft")

In [7]:
# You can get basic descriptive stats about the network 
feed.describe()

,indicator,value
0,agencies,"[Cape Cod Regional Transit Authority, MBTA]"
1,timezone,America/New_York
2,start_date,20260302
3,end_date,20260404
4,num_routes,397
5,num_trips,55165
6,num_stops,10244
7,num_shapes,1117
8,sample_date,20260305
9,num_routes_active_on_sample_date,176


In [42]:
# All of the same files are read in
feed.stops.head()

,stop_id,stop_code,stop_name,stop_desc,platform_code,platform_name,stop_lat,stop_lon,zone_id,stop_address,stop_url,level_id,location_type,parent_station,wheelchair_boarding,municipality,on_street,at_street,vehicle_type
0,1,1,Washington St opp Ruggles St,<NA>,<NA>,<NA>,42.330957,-71.082754,ExpressBus-Downtown,<NA>,https://www.mbta.com/stops/1,<NA>,0,<NA>,1,Boston,Washington Street,Ruggles Street,3
1,10,10,Theo Glynn Way @ Newmarket Sq,<NA>,<NA>,<NA>,42.330555,-71.068787,LocalBus,<NA>,https://www.mbta.com/stops/10,<NA>,0,<NA>,1,Boston,Theodore Glynn Way,Newmarket Square,3
2,10000,10000,Tremont St opp Temple Pl,<NA>,<NA>,<NA>,42.355692,-71.062911,LocalBus,<NA>,https://www.mbta.com/stops/10000,<NA>,0,<NA>,1,Boston,Tremont Street,Temple Place,3
3,10003,10003,Albany St opp Randall St,<NA>,<NA>,<NA>,42.331591,-71.076237,LocalBus,<NA>,https://www.mbta.com/stops/10003,<NA>,0,<NA>,1,Boston,Albany Street,Randall Street,3
4,10005,10005,Albany St opp E Concord St,<NA>,<NA>,<NA>,42.335017,-71.071280,LocalBus,<NA>,https://www.mbta.com/stops/10005,<NA>,0,<NA>,1,Boston,Albany Street,<NA>,3


In [43]:
feed.routes.head()

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color,route_sort_order,route_fare_class,line_id,listed_route,network_id
0,Red,1,<NA>,Red Line,Rapid Transit,1,https://www.mbta.com/schedules/Red,DA291C,FFFFFF,10010,Rapid Transit,line-Red,<NA>,rapid_transit
1,Mattapan,1,<NA>,Mattapan Line,Rapid Transit,0,https://www.mbta.com/schedules/Mattapan,DA291C,FFFFFF,10011,Rapid Transit,line-Mattapan,<NA>,m_rapid_transit
2,Orange,1,<NA>,Orange Line,Rapid Transit,1,https://www.mbta.com/schedules/Orange,ED8B00,FFFFFF,10020,Rapid Transit,line-Orange,<NA>,rapid_transit
3,Green-B,1,B,Green Line B,Rapid Transit,0,https://www.mbta.com/schedules/Green-B,00843D,FFFFFF,10032,Rapid Transit,line-Green,<NA>,rapid_transit
4,Green-C,1,C,Green Line C,Rapid Transit,0,https://www.mbta.com/schedules/Green-C,00843D,FFFFFF,10033,Rapid Transit,line-Green,<NA>,rapid_transit


In [44]:
feed.shapes.head()

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,010140,42.373040,-71.117686,10001,NaN
1,010140,42.373096,-71.117908,10002,NaN
2,010140,42.373269,-71.118463,10003,NaN
3,010140,42.373353,-71.118582,10004,NaN
4,010140,42.373237,-71.118760,10005,NaN


In [45]:
feed.trips.head()

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,trip_route_type,route_pattern_id,bikes_allowed
0,1,BUS12026-hbc16fr1-Weekday-02,73542513,Harvard,<NA>,0,C01-7,010154,1,<NA>,1-_-0,1
1,1,BUS12026-hbc16fr1-Weekday-02,73542518,Harvard,<NA>,0,C01-8,010154,1,<NA>,1-_-0,1
2,1,BUS12026-hbc16fr1-Weekday-02,73542519,Harvard,<NA>,0,C01-9,010154,1,<NA>,1-_-0,1
3,1,BUS12026-hbc16fr1-Weekday-02,73542520,Harvard,<NA>,0,C01-7,010154,1,<NA>,1-_-0,1
4,1,BUS12026-hbc16fr1-Weekday-02,73542522,Harvard,<NA>,0,C01-2,010154,1,<NA>,1-_-0,1


In [46]:
feed.stop_times.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,timepoint,checkpoint_id,continuous_pickup,continuous_drop_off
0,72758838,06:00:00,06:00:00,70036,1,<NA>,0,1,0,ogmnl,<NA>,<NA>
1,72758838,06:01:00,06:01:00,70034,10,<NA>,0,0,0,mlmnl,<NA>,<NA>
2,72758838,06:05:00,06:05:00,70032,20,<NA>,0,0,0,welln,<NA>,<NA>
3,72758838,06:07:00,06:07:00,70278,30,<NA>,0,0,0,astao,<NA>,<NA>
4,72758838,06:09:00,06:09:00,70030,40,<NA>,0,0,0,sull,<NA>,<NA>


In [58]:
## YOUR TURN 
## Determine how many unique trips are taken on the "Red" route in Boston



## 3 Connecting between the files

What if you want to know the stops of a transit line? Or maybe just the subway systems? You need to get from routes to stops via the intermediate paths. 

Recall, here are some important connector columns between the files: 
* `route_id`: connects `routes` and `trips`
* `trip_id`: connects `trips` and `stop_times`
* `shape_id`: connects `shapes` and `trips`
* `stop_id`: connects `stops` and `stop_times`


So, to get geometry values for the routes, we need to follow the path:

`routes:route_id` $\rightarrow$ `trips:route_id` 

`trips:trip_id` $\rightarrow$ `stop_times:trip_id` 

`stop_times:stop_id` $\rightarrow$ `stops:stop_id`

In [59]:
# Let's look at routes again
routes.head()

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color,route_sort_order,route_fare_class,line_id,listed_route,network_id
0,Red,1,NaN,Red Line,Rapid Transit,1,https://www.mbta.com/schedules/Red,DA291C,FFFFFF,10010,Rapid Transit,line-Red,NaN,rapid_transit
1,Mattapan,1,NaN,Mattapan Line,Rapid Transit,0,https://www.mbta.com/schedules/Mattapan,DA291C,FFFFFF,10011,Rapid Transit,line-Mattapan,NaN,m_rapid_transit
2,Orange,1,NaN,Orange Line,Rapid Transit,1,https://www.mbta.com/schedules/Orange,ED8B00,FFFFFF,10020,Rapid Transit,line-Orange,NaN,rapid_transit
3,Green-B,1,B,Green Line B,Rapid Transit,0,https://www.mbta.com/schedules/Green-B,00843D,FFFFFF,10032,Rapid Transit,line-Green,NaN,rapid_transit
4,Green-C,1,C,Green Line C,Rapid Transit,0,https://www.mbta.com/schedules/Green-C,00843D,FFFFFF,10033,Rapid Transit,line-Green,NaN,rapid_transit


In [60]:
# What types of transit are there? 
routes.route_desc.value_counts()

route_desc
Rail Replacement Bus    214
Local Bus               106
Frequent Bus             23
Commuter Bus             15
Regional Rail            14
Rapid Transit             8
Coverage Bus              7
Seasonal Ferry            5
Supplemental Bus          3
Ferry                     2
Name: count, dtype: int64

In [62]:
# First, let's keep only the rapid transit lines
major_routes = routes[routes.route_desc == 'Rapid Transit']

# Then let's connect it to trips via route_id
major_trips = trips[trips['route_id'].isin(major_routes['route_id'].unique())]
print(len(major_trips))
major_trips.head()

13174


,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,trip_route_type,route_pattern_id,bikes_allowed
40278,Blue,RTL12026-hmb16bf1-Weekday-01,74361927,Bowdoin,NaN,0,B946_-1,946_0013,1,NaN,Blue-6-0,0
40279,Blue,RTL12026-hmb16bf1-Weekday-01,74361929,Bowdoin,NaN,0,B946_-2,946_0013,1,NaN,Blue-6-0,0
40280,Blue,RTL12026-hmb16bf1-Weekday-01,74361931,Bowdoin,NaN,0,B946_-5,946_0013,1,NaN,Blue-6-0,0
40281,Blue,RTL12026-hmb16bf1-Weekday-01,74361933,Bowdoin,NaN,0,B946_-6,946_0013,1,NaN,Blue-6-0,0
40282,Blue,RTL12026-hmb16bf1-Weekday-01,74361935,Bowdoin,NaN,0,B946_-7,946_0013,1,NaN,Blue-6-0,0


In [63]:
# Now let's connect trips to stop_times via trip_id
major_stop_times = stop_times[stop_times['trip_id'].isin(major_trips['trip_id'].values)]
len(major_stop_times)
major_stop_times.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,timepoint,checkpoint_id,continuous_pickup,continuous_drop_off
1344287,74360480,05:12:00,05:12:00,70059,1,NaN,0,1,0,wondl,NaN,NaN
1344288,74360480,05:13:00,05:13:00,70057,10,NaN,0,0,0,rbmnl,NaN,NaN
1344289,74360480,05:15:00,05:15:00,70055,20,NaN,0,0,0,bmmnl,NaN,NaN
1344290,74360480,05:16:00,05:16:00,70053,30,NaN,0,0,0,sdmnl,NaN,NaN
1344291,74360480,05:18:00,05:18:00,70051,40,NaN,0,0,0,orhte,NaN,NaN


In [64]:
# Finally, let's connect stop_times to stops via stop_id
major_stops = stops[stops['stop_id'].isin(stop_times['stop_id'].values)]
len(major_stops)
major_stops.head()

,stop_id,stop_code,stop_name,stop_desc,platform_code,platform_name,stop_lat,stop_lon,zone_id,stop_address,stop_url,level_id,location_type,parent_station,wheelchair_boarding,municipality,on_street,at_street,vehicle_type
2,10000,10000.0,Tremont St opp Temple Pl,NaN,NaN,NaN,42.355692,-71.062911,LocalBus,NaN,https://www.mbta.com/stops/10000,NaN,0,NaN,1,Boston,Tremont Street,Temple Place,3.0
22,102,102.0,Massachusetts Ave @ Prospect St,NaN,NaN,NaN,42.365291,-71.103404,LocalBus,NaN,https://www.mbta.com/stops/102,NaN,0,NaN,1,Cambridge,Massachusetts Avenue,Essex Street,3.0
179,110,110.0,Massachusetts Ave opp Holyoke St,NaN,NaN,NaN,42.373111,-71.117653,LocalBus,NaN,https://www.mbta.com/stops/110,NaN,0,NaN,1,Cambridge,Massachusetts Avenue,Holyoke Street,3.0
228,11366,11366.0,Pearl St @ Brookline Village,NaN,NaN,NaN,42.332409,-71.116433,LocalBus,NaN,https://www.mbta.com/stops/11366,NaN,0,NaN,1,Brookline,Pearl Street,NaN,3.0
231,11384,11384.0,Dartmouth St @ Back Bay Sta,NaN,NaN,NaN,42.347346,-71.075956,ExpressBus-Downtown,NaN,https://www.mbta.com/stops/11384,NaN,0,NaN,1,Boston,Dartmouth Street,Stuart Street,3.0


In [65]:
## YOUR TURN 
## Turn major_stops into a geodataframe and plot the location of the stops


# Network Review and Practice

Let's put both of our pieces together now. 

Your goal is to locate the nearest rapid transit station in Boston, Massachusetts from the following locations, then map out the shortest (by distance) walking path to get there using `osmnx`. 

Think about the steps you will need to go through:
* Identify the rapid transit stations
* Find the minimum distance between our location of interest and the stations
* Pull the walking network for Boston
* Get the closest nodes to the origin (location) and destination (station)
* Plot the shortest path

Location 1: Grocery store at lon: `-71.011028`, lat: `42.427241`

Location 2: Beach at lon: `-71.016152`, lat: `42.280145`